In [ ]:
from sklearn.feature_extraction import DictVectorizer
import xgboost as xgb
from sklearn.pipeline import Pipeline
import pickle

In [32]:
import pandas as pd
import matplotlib.pyplot as plt
from pprint import pprint
import math
pd.__version__

'2.3.3'

In [33]:
import os
import mlflow

os.environ['AWS_PROFILE'] = 'sergio'

TRACKING_URI = '13.61.191.33'
mlflow.set_tracking_uri(f'http://{TRACKING_URI}:5000')
mlflow.set_experiment('nyc-taxi-experiment-deployment')

<Experiment: artifact_location='s3://mlflow-artifacts-remote-786875389232/38', creation_time=1794920439902, effective_trace_archival_retention=None, experiment_id='38', last_update_time=1794920439902, lifecycle_stage='active', name='nyc-taxi-experiment-deployment', tags={}, trace_location=None, workspace='default'>

In [34]:
def read_data(file):
    df = pd.read_parquet(file)
    print(f"Number of columns for {file}: {len(df.columns)}")
    # print(df.dtypes)
    #convert to datetime64[ns] to int64
    converted_start_end_df = df.astype({"lpep_pickup_datetime": "int", "lpep_dropoff_datetime": "int"})
    df['duration'] = (converted_start_end_df['lpep_dropoff_datetime'] - converted_start_end_df['lpep_pickup_datetime']) * 1.6666666666667e-8
    # print(df.duration.describe())
    print(f"min: {df.duration.min()}")
    print(f"max: {df.duration.max()}")
    df_b060 = df[(df.duration >= 1) & (df.duration <= 60)]
    # Q:What fraction of the records left after you dropped the outliers?
    fraction = len(df_b060) / len(df)
    print(f"fraction: {fraction*100:.2f}")
    return df_b060[["PULocationID", "DOLocationID", "duration"]]

In [35]:
train_df = read_data("https://d37ci6vzurychx.cloudfront.net/trip-data/green_tripdata_2021-01.parquet")
print("done reading train data")

Number of columns for https://d37ci6vzurychx.cloudfront.net/trip-data/green_tripdata_2021-01.parquet: 20
min: 0.0
max: 1439.6000000000288
fraction: 96.54
done reading train data


In [36]:
val_df = read_data("https://d37ci6vzurychx.cloudfront.net/trip-data/green_tripdata_2021-02.parquet")
print("done reading val data")

Number of columns for https://d37ci6vzurychx.cloudfront.net/trip-data/green_tripdata_2021-02.parquet: 20
min: 0.0
max: 1439.6166666666957
fraction: 95.80
done reading val data


In [37]:
# sns.distplot(train_df.duration, label="train")
# sns.distplot(val_df.duration, label="val")
# plt.legend()

In [38]:
categorical = ["PULocationID", "DOLocationID"]
numerical = ['trip_distance']

vectorizer = DictVectorizer()

train_dicts = train_df[categorical].astype(str).to_dict(orient='records')
pprint(train_dicts[:5])
X_train = vectorizer.fit_transform(train_dicts)
del train_dicts

[{'DOLocationID': '151', 'PULocationID': '43'},
 {'DOLocationID': '239', 'PULocationID': '166'},
 {'DOLocationID': '42', 'PULocationID': '41'},
 {'DOLocationID': '75', 'PULocationID': '168'},
 {'DOLocationID': '75', 'PULocationID': '75'}]


In [39]:
val_dicts = val_df[categorical].astype(str).to_dict(orient='records')
pprint(val_dicts[:5])
X_val = vectorizer.transform(val_dicts)
del val_dicts

[{'DOLocationID': '205', 'PULocationID': '130'},
 {'DOLocationID': '244', 'PULocationID': '152'},
 {'DOLocationID': '48', 'PULocationID': '152'},
 {'DOLocationID': '241', 'PULocationID': '152'},
 {'DOLocationID': '42', 'PULocationID': '75'}]


In [40]:
print(X_val.shape)

(61859, 506)


In [41]:
target = "duration"
y_train = train_df[target].values
y_val = val_df[target].values

train = xgb.DMatrix(X_train, label=y_train)
validation = xgb.DMatrix(X_val, label=y_val)

In [42]:
del train_df
del val_df

In [43]:
# def objective(params):
#     with mlflow.start_run():
#         mlflow.set_tag("developer", "sergio")
#         mlflow.log_param("model", "xgboost")
#         mlflow.log_params(params)
#         booster = xgb.train(
#             params=params,
#             dtrain=train,
#             num_boost_round=5,
#             evals=[(validation, "validation")],
#             early_stopping_rounds=10
#         )
#         y_pred = booster.predict(validation)
#         rmse = root_mean_squared_error(y_val, y_pred)
#         mlflow.log_metric("rmse", rmse)
#     return {"loss": rmse, "status": STATUS_OK}

# search_space = {
#     'max_depth': scope.int(hp.quniform('max_depth', 4, 100, 1)),
#     'learning_rate': hp.loguniform('learning_rate', -3, 0),
#     'reg_alpha': hp.loguniform('reg_alpha', -5, -1),
#     'reg_lambda': hp.loguniform('reg_lambda', -6, -1),
#     'min_child_weight': hp.loguniform('min_child_weight', -1, 3),
#     'objective': 'reg:linear',
#     'seed': 42
# }

# best_result = fmin(
#     fn=objective,
#     space=search_space,
#     algo=tpe.suggest,
#     max_evals=50,
#     trials=Trials()
# )



In [44]:
# print("Best hyperparameters:", best_result)

In [45]:
import numpy as np

mlflow.autolog()
with mlflow.start_run():
    mlflow.set_tag("developer", "sergio")
    best_result = {'learning_rate': np.float64(0.20844413031306103), 
                   'max_depth': np.int64(99.0), 
                   'min_child_weight': np.float64(1.5432280770679585), 
                   'reg_alpha': np.float64(0.04953891913131428), 
                   'reg_lambda': np.float64(0.017060826809340544)}
    booster = xgb.train(
        params=best_result,
        dtrain=train,
        num_boost_round=2,
        evals=[(validation, "validation")],
        early_stopping_rounds=10
    )
    y_pred = booster.predict(validation)
    with open("models/preprocessor.b", "wb") as f_out:
        pickle.dump(vectorizer, f_out)
    mlflow.log_artifact("models/preprocessor.b", artifact_path="preprocessor")


2026/11/17 14:20:06 INFO mlflow.bedrock: Enabled auto-tracing for Bedrock. Note that MLflow can only trace boto3 service clients that are created after this call. If you have already created one, please recreate the client by calling `boto3.client`.
2026/11/17 14:20:06 INFO mlflow.tracking.fluent: Autologging successfully enabled for boto3.
2026/11/17 14:20:06 INFO mlflow.tracking.fluent: Autologging successfully enabled for sklearn.
2026/11/17 14:20:06 INFO mlflow.tracking.fluent: Autologging successfully enabled for xgboost.


[0]	validation-rmse:11.10610
[1]	validation-rmse:10.34909


2026/11/17 14:20:07 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


🏃 View run polite-midge-654 at: http://13.61.191.33:5000/#/experiments/38/runs/269683ecb9a54ee5ac6ea3e01c4f7806
🧪 View experiment at: http://13.61.191.33:5000/#/experiments/38
